# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$


## Parte 1 — Modela el MDP

Se completa la clase `WarehouseMDP` con las convenciones de la izquierda/derecha *relativas a la dirección de movimiento* (giro, no coordenadas absolutas del grid).

In [ ]:
import numpy as np

UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)

class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slip_intended=0.60):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterias / paredes (a partir de la imagen)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales: {(row, col): reward}
        self.terminal_states = {
            (0, 5): 10.0,   # ENTREGA
            (2, 2): 2.0,    # CARGA (estacion de carga)
            (3, 5): -10.0,  # PELIGRO MORTAL
        }

        # Peligros NO terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma
        self.slip_intended = slip_intended  # prob. de moverse en la direccion deseada en piso resbaloso

        self.actions = [UP, DOWN, LEFT, RIGHT]

        # Desviaciones (izquierda, derecha) RELATIVAS a la direccion en la que se mueve el robot
        self.perp = {
            UP:    (LEFT, RIGHT),
            DOWN:  (RIGHT, LEFT),
            LEFT:  (DOWN, UP),
            RIGHT: (UP, DOWN),
        }

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...]
        - Un estado terminal es absorbente: se queda en si mismo con prob 1.
        - Las probabilidades dependen de si `state` es resbaloso.
        - Si el movimiento sale del grid o golpea una pared, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = self.slip_intended
            p_dev = (1.0 - p_intended) / 2.0
        else:
            p_intended, p_dev = 0.90, 0.05

        left_dev, right_dev = self.perp[action]
        outcomes = [(action, p_intended), (left_dev, p_dev), (right_dev, p_dev)]

        result = {}
        for act, prob in outcomes:
            ns = (state[0] + act[0], state[1] + act[1])
            if not self.is_valid_state(ns):
                ns = state
            result[ns] = result.get(ns, 0.0) + prob

        return list(result.items())


### Validación mínima del modelo

Antes de implementar Bellman, se valida primero el MDP.

In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Numero de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("Todas las distribuciones de transicion suman 1.")


Numero de estados: 26
Todas las distribuciones de transicion suman 1.


## Parte 2 — Value Iteration

$$V_{k+1}(s)=R(s)+\gamma\max_a\sum_{s'}T(s,a,s')V_k(s')$$


In [ ]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[ns] for ns, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
                V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, i + 1
    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = {a: expected_next_value(grid, s, a, V) for a in grid.actions}
        policy[s] = max(q_values, key=q_values.get)
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation
$$V_{k+1}^{\pi}(s)=R(s)+\gamma\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')$$

### Policy Improvement
$$\pi_{\mathrm{new}}(s)=\arg\max_a\sum_{s'}T(s,a,s')V^\pi(s')$$


In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = {a: expected_next_value(grid, s, a, V) for a in grid.actions}
        policy[s] = max(q_values, key=q_values.get)
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria (siempre UP)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for i in range(max_iter):
        # 2. evaluacion
        V = policy_evaluation(grid, policy, threshold=threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        changed = sum(1 for s in new_policy if new_policy[s] != policy[s])
        history.append(changed)
        policy = new_policy
        # 4. repetir hasta estabilidad
        if changed == 0:
            break
    return policy, V, history


## Parte 4 — Visualización y comparación

In [ ]:
ARROWS = {
    (-1, 0): "\u2191",
    ( 1, 0): "\u2193",
    ( 0,-1): "\u2190",
    ( 0, 1): "\u2192",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolitica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolitica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\nAmbos algoritmos encontraron la misma politica optima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  

## Parte 5 — Interpreta la política

**1. ¿Busca la entrega +10 o la carga +2?**

Va a CARGA. Me sorprendió porque el +10 es mayor, pero el robot ni se acerca a la entrega.

**2. ¿Por qué una recompensa menor puede ser óptima?**

Porque para llegar a la entrega no queda otra que cruzar la celda de peligro (-3), ya que las estanterías tapan cualquier otro camino. Sumando esos pasos, el -3 y el descuento (γ=0.9), el +10 termina valiendo menos que un +2 cercano y sin riesgo.

**3. ¿Dónde cambia el piso resbaloso la decisión?**

En ningún estado cambia la ruta, porque las celdas resbalosas son el único camino posible hacia las metas. Lo que sí hacen es bajar un poco el valor de esos estados frente a un piso normal equivalente.

**4. ¿Qué papel cumple el -1 por paso?**

Le mete urgencia: hace que la distancia importe. Sin ese costo, dar más pasos no tendría penalización y probablemente convendría ir por el +10.

**5. ¿Por qué T ya no es igual para todos los estados?**

Porque las probabilidades de moverse dependen de si el estado actual es resbaloso o no (0.90/0.05/0.05 vs 0.60/0.20/0.20), entonces T tiene que revisar en qué tipo de piso está el robot antes de definir la distribución.


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

Predicción: pensé que con pasos casi gratis el robot se animaría a ir por la entrega.

Resultado: no cambió, sigue yendo a CARGA. El -3 de la celda de peligro no depende de `living_reward`, así que bajarlo no ayuda a la entrega.


In [ ]:
grid_A = WarehouseMDP(living_reward=-0.1)
V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)
print("Iteraciones:", n_A)
print_values(grid_A, V_A)
print()
print_policy(grid_A, pi_A)
print("\nV(start) =", V_A[grid_A.start])


Iteraciones: 26
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = 1.6487099915120942


### Experimento B — Piso muy resbaloso (`0.60 → 0.40`)

Predicción: con más chance de desviarse pensé que podría cambiar la ruta.

Resultado: la política queda igual, solo bajan un poco los valores — no hay otro camino que evite las celdas resbalosas.


In [ ]:
grid_B = WarehouseMDP(slip_intended=0.40)
V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)
print("Iteraciones:", n_B)
print_values(grid_B, V_B)
print()
print_policy(grid_B, pi_B)
print("\nV(start) =", V_B[grid_B.start])


Iteraciones: 22
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = -2.706200866141053


### Experimento C — Más paciencia (`gamma = 0.99`)

Predicción: con menos descuento pensé que el +10 se volvería más competitivo.

Resultado: desde `START` sigue yendo a CARGA, aunque en (1,2) la flecha sí cambia respecto al caso base. El -3 del camino a la entrega sigue pesando más que la paciencia extra.


In [ ]:
grid_C = WarehouseMDP(gamma=0.99)
V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)
print("Iteraciones:", n_C)
print_values(grid_C, V_C)
print()
print_policy(grid_C, pi_C)
print("\nV(start) =", V_C[grid_C.start])


Iteraciones: 24
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = -1.456064938873927


### Bonus — ¿Hay un `living_reward` donde `START` cambia a ENTREGA?

Barrí `living_reward` de -3 a +3.5 viendo la política en (0,2).


In [ ]:
prev = None
for lr in np.arange(-3.0, 3.51, 0.1):
    g = WarehouseMDP(living_reward=lr)
    V, n = value_iteration(g)
    pi = extract_policy(g, V)
    a = ARROWS[pi[(0, 2)]]
    if a != prev:
        print(f"living_reward={lr:+.2f}  politica en (0,2) = {a}   V(0,2) = {V[(0,2)]:.3f}")
        prev = a


living_reward=-3.00  politica en (0,2) = ↓   V(0,2) = -5.858


living_reward=+2.00  politica en (0,2) = ↑   V(0,2) = 19.999


living_reward=+3.20  politica en (0,2) = ←   V(0,2) = 31.999
living_reward=+3.30  politica en (0,2) = ↑   V(0,2) = 32.999


**No logré encontrar un umbral de `living_reward` donde la política cambie a favor de la entrega.** Probé desde -3.0 hasta +3.5 en pasos de 0.1 y la política en (0,2) se mantiene igual (yendo hacia carga) en casi todo el rango; solo cambia cuando `living_reward` se pone positivo y bastante grande (como +2 en adelante), pero ahí ya no tiene mucho sentido físico porque estaría "premiando" moverse en vez de costarle algo al robot, así que no creo que cuente como respuesta real a la pregunta.

Mi sospecha es que el -3 de la celda de peligro que hay que cruzar para llegar a la entrega es un costo fijo que no depende de `living_reward`, entonces por más que baje el costo por paso, ese -3 sigue estando ahí y no deja que la entrega compita con la carga. Pero no alcancé a comprobar esto del todo (habría que probar cambiando otras cosas, como el valor de ese peligro, para confirmarlo). Dejo el barrido de todas formas para mostrar que sí lo intenté.
